# Session 03: Markov Decision Process & Reinforcement Learning

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL4 + TL5)  
**Level**: 🟢 Beginner → 🟡 Intermediate → 🔴 Advanced

---

## 📋 What You'll Learn

| Section | Topic | Level |
|---------|-------|---------|
| 1 | What is Reinforcement Learning? | 🟢 |
| 2 | The RL Framework & Key Components | 🟢 |
| 3 | Markov Decision Processes (MDPs) | 🟢–🟡 |
| 4 | Q-Learning (Value-Based RL) | 🟡 |
| 5 | SARSA (On-Policy Learning) | 🟡 |
| 6 | Policy Gradient Methods | 🔴 |
| 7 | Multi-Armed Bandits & Exploration | 🟢–🟡 |
| 8 | Real-World RL Applications | 🟢 |

In [ ]:
# Setup — Run this cell first!
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('✅ All imports successful!')
print('   NumPy version:', np.__version__)

---

## §1 — What is Reinforcement Learning? 🟢

**Reinforcement Learning (RL)** is the third major paradigm of machine learning (alongside supervised and unsupervised). In RL, an **agent** learns by **interacting** with an **environment**, receiving **rewards** or **penalties** for its actions.

### How RL Differs from Other ML Types

| Type | Data | Learning Signal | Example |
|------|------|----------------|--------|
| **Supervised** | Labeled (X → Y) | Correct answers | Email spam detection |
| **Unsupervised** | Unlabeled (X only) | No feedback | Customer clustering |
| **Reinforcement** | Sequential interactions | Delayed rewards | Game-playing AI |

### Real-World Analogy: Training a Dog 🐕

- **Agent**: The dog
- **Environment**: Your house
- **Actions**: Sit, stay, fetch, bark
- **Reward**: Treats for good behavior (+1)
- **Penalty**: Scolding for bad behavior (-1)
- **Policy**: The dog's learned behavior strategy

In [ ]:
# §1: The Fundamental RL Loop
# Every RL algorithm follows this basic pattern:

def rl_training_loop(agent, environment, n_episodes=100):
    """The fundamental RL training loop — foundation of ALL RL."""
    all_rewards = []
    
    for episode in range(n_episodes):
        state = environment.reset()          # 1. Start fresh
        total_reward = 0
        done = False
        
        while not done:
            action = agent.choose_action(state)     # 2. Agent decides
            next_state, reward, done = environment.step(action)  # 3. Environment responds
            agent.learn(state, action, reward, next_state)       # 4. Agent learns
            
            state = next_state
            total_reward += reward
        
        all_rewards.append(total_reward)
    
    return all_rewards

print("The RL loop: Reset → Act → Observe → Learn → Repeat")
print("This is the skeleton of Q-Learning, SARSA, Policy Gradient, and more!")

---

## §2 — The RL Framework & Key Components 🟢

```
            ┌──────────────────┐
            │   ENVIRONMENT    │
    Action  │                  │  Reward
   ────────►│   State: s_t     │◄────────
            │   Next State:    │
            │     s_{t+1}      │
            └────────┬─────────┘
                     │ Observation
                     ▼
            ┌──────────────────┐
            │      AGENT       │
            │   Policy: π(s)   │
            │   "What to do    │
            │    in state s"   │
            └──────────────────┘
```

In [ ]:
# §2: RL Components Summary

components = {
    'State (s)': 'Current situation the agent observes',
    'Action (a)': 'What the agent decides to do',
    'Reward (r)': 'Immediate feedback from the environment',
    'Policy π(s)': 'Strategy: which action to take in each state',
    'Value V(s)': 'Expected total future reward from state s',
    'Q-value Q(s,a)': 'Expected total future reward for action a in state s',
    'Discount γ': 'How much to value future vs immediate rewards (0-1)',
}

print("┌─────────────────────────────────────────────────────────────────┐")
print("│  Key RL Components                                             │")
print("├────────────────┬────────────────────────────────────────────────┤")
for name, desc in components.items():
    print(f"│ {name:<14s} │ {desc:<42s}   │")
print("└────────────────┴────────────────────────────────────────────────┘")

# Interactive: The Discount Factor
print("\n📊 Understanding the Discount Factor (γ):")
print("─" * 50)
for gamma in [0.1, 0.5, 0.9, 0.99]:
    # Future reward of 100, received after 10 steps
    discounted = 100 * (gamma ** 10)
    print(f"  γ={gamma:.2f}: $100 in 10 steps is worth ${discounted:>6.2f} now")
    
print("\n  → High γ = patient agent (values future rewards)")
print("  → Low γ  = greedy agent (wants rewards NOW)")

---

## §3 — Markov Decision Processes (MDPs) 🟢–🟡

An **MDP** is the mathematical framework underlying RL. It formalizes the problem the agent must solve.

### MDP = (S, A, P, R, γ)

| Symbol | Name | Meaning |
|--------|------|---------|
| **S** | State space | All possible situations |
| **A** | Action space | All possible actions |
| **P(s'\|s,a)** | Transition probability | Prob. of reaching s' from s via action a |
| **R(s,a)** | Reward function | Immediate reward for action a in state s |
| **γ** | Discount factor | How much to value future rewards |

### The Markov Property

> **"The future depends only on the present, not the past."**
>
> The current state captures ALL information needed for decision-making.

In [ ]:
# §3: Build a Simple GridWorld MDP

class GridWorld:
    """
    4x4 GridWorld — Our test environment.
    
    ┌─────┬─────┬─────┬─────┐
    │  S  │  .  │  .  │  .  │
    ├─────┼─────┼─────┼─────┤
    │  .  │  ■  │  .  │  T  │  S = Start
    ├─────┼─────┼─────┼─────┤  ■ = Wall
    │  .  │  .  │  .  │  .  │  T = Trap (-1)
    ├─────┼─────┼─────┼─────┤  ★ = Goal (+1)
    │  .  │  ■  │  .  │  ★  │
    └─────┴─────┴─────┴─────┘
    """
    
    def __init__(self):
        self.grid_size = 4
        self.n_states = 16
        self.n_actions = 4  # LEFT=0, DOWN=1, RIGHT=2, UP=3
        self.action_names = ['←', '↓', '→', '↑']
        
        self.start = 0
        self.goal = 15
        self.traps = {7}
        self.walls = {5, 13}
        self.state = self.start
        
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.grid_size)
        
        if action == 0:   new_row, new_col = row, max(0, col-1)        # LEFT
        elif action == 1: new_row, new_col = min(3, row+1), col        # DOWN
        elif action == 2: new_row, new_col = row, min(3, col+1)        # RIGHT
        elif action == 3: new_row, new_col = max(0, row-1), col        # UP
        
        new_state = new_row * self.grid_size + new_col
        
        # Walls: bounce back
        if new_state in self.walls:
            new_state = self.state
        
        self.state = new_state
        
        if self.state == self.goal:
            return self.state, 1.0, True
        elif self.state in self.traps:
            return self.state, -1.0, True
        else:
            return self.state, -0.04, False

# Test the environment
env = GridWorld()
state = env.reset()
print(f"Start state: {state}")

# Take some actions
actions_demo = [2, 1, 2, 1, 2, 1]  # RIGHT, DOWN, RIGHT, DOWN, RIGHT, DOWN
print(f"\nWalking through the grid:")
for action in actions_demo:
    next_state, reward, done = env.step(action)
    print(f"  Action: {env.action_names[action]} → State: {next_state}, "
          f"Reward: {reward:+.2f}, Done: {done}")
    if done:
        break

In [ ]:
# §3: Visualize the GridWorld

def visualize_gridworld(env, Q=None, title="GridWorld"):
    """Visualize the GridWorld with optional Q-values."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Draw grid
    for i in range(5):
        ax.axhline(y=i, color='black', linewidth=1)
        ax.axvline(x=i, color='black', linewidth=1)
    
    # Color cells
    for row in range(4):
        for col in range(4):
            s = row * 4 + col
            
            if s == env.start:
                ax.add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#3498db', alpha=0.3))
                ax.text(col+0.5, 3-row+0.5, 'S', ha='center', va='center', fontsize=16, fontweight='bold')
            elif s == env.goal:
                ax.add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#2ecc71', alpha=0.3))
                ax.text(col+0.5, 3-row+0.5, '★', ha='center', va='center', fontsize=20)
            elif s in env.traps:
                ax.add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#e74c3c', alpha=0.3))
                ax.text(col+0.5, 3-row+0.5, 'T', ha='center', va='center', fontsize=16, fontweight='bold', color='red')
            elif s in env.walls:
                ax.add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#2c3e50', alpha=0.8))
                ax.text(col+0.5, 3-row+0.5, '■', ha='center', va='center', fontsize=16, color='white')
            elif Q is not None:
                best_action = np.argmax(Q[s])
                arrow = env.action_names[best_action]
                ax.text(col+0.5, 3-row+0.5, arrow, ha='center', va='center', fontsize=18, fontweight='bold', color='#2c3e50')
    
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(np.arange(0.5, 4, 1))
    ax.set_yticks(np.arange(0.5, 4, 1))
    ax.set_xticklabels(range(4))
    ax.set_yticklabels(range(3, -1, -1))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    plt.tight_layout()
    plt.show()

visualize_gridworld(env, title="GridWorld Environment")
print("S = Start  |  ★ = Goal (+1)  |  T = Trap (-1)  |  ■ = Wall")

---

## §4 — Q-Learning (Value-Based RL) 🟡

**Q-Learning** learns a **Q-table** that maps each (state, action) pair to its expected future reward.

### Update Rule:

```
Q(s, a) ← Q(s, a) + α × [r + γ × max Q(s', a') – Q(s, a)]
                          └────────── TD Target ──────────┘
                     └──────────────── TD Error ──────────────┘
```

- **α** = learning rate (how fast to update)
- **γ** = discount factor (how much to value future)
- **max Q(s', a')** = best Q-value from the next state (**off-policy**: uses the BEST possible action)

In [ ]:
# §4: Q-Learning Implementation

def q_learning(env, n_episodes=10000, alpha=0.1, gamma=0.99,
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995):
    """
    Q-Learning — the most popular RL algorithm.
    Off-policy: learns the optimal policy regardless of behavior.
    """
    Q = np.zeros((env.n_states, env.n_actions))
    epsilon = epsilon_start
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            # ε-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(env.n_actions)  # EXPLORE
            else:
                action = np.argmax(Q[state])               # EXPLOIT
            
            next_state, reward, done = env.step(action)
            
            # Q-Learning update
            best_next = np.max(Q[next_state])
            td_target = reward + gamma * best_next * (1 - done)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state = next_state
            total_reward += reward
            steps += 1
        
        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, rewards_history

# Train!
env = GridWorld()
Q_learned, rewards = q_learning(env, n_episodes=10000)

print("✅ Q-Learning Training Complete!")
print(f"   Episodes: 10,000")
print(f"   Final avg reward (last 100): {np.mean(rewards[-100:]):.3f}")
print(f"   Success rate (last 500): {sum(1 for r in rewards[-500:] if r > 0) / 500:.1%}")

In [ ]:
# §4: Visualize the Q-Learning Results

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Q-Learning Results on GridWorld', fontsize=16, fontweight='bold')

# 1. Learning Curve
window = 100
smoothed = np.convolve(rewards, np.ones(window)/window, 'valid')
axes[0].plot(smoothed, linewidth=1.5, color='#3498db')
axes[0].set_title('Learning Curve')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward (100-ep avg)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# 2. Q-Value Heatmap
q_max = np.max(Q_learned, axis=1).reshape(4, 4)
for w in env.walls:
    r, c = divmod(w, 4)
    q_max[r, c] = np.nan
im = axes[1].imshow(q_max, cmap='RdYlGn', interpolation='nearest')
axes[1].set_title('State Values (max Q)')
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        if s in env.walls:
            axes[1].text(j, i, '■', ha='center', va='center', fontsize=14)
        elif s == env.goal:
            axes[1].text(j, i, '★', ha='center', va='center', fontsize=14)
        elif s in env.traps:
            axes[1].text(j, i, 'T', ha='center', va='center', fontsize=14)
        elif not np.isnan(q_max[i, j]):
            axes[1].text(j, i, f'{q_max[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[1])

# 3. Learned Policy
for i in range(5):
    axes[2].axhline(y=i, color='black', linewidth=0.5)
    axes[2].axvline(x=i, color='black', linewidth=0.5)
for row in range(4):
    for col in range(4):
        s = row * 4 + col
        if s == env.start:
            axes[2].text(col+0.5, 3-row+0.5, 'S', ha='center', va='center', fontsize=14, fontweight='bold', color='blue')
        elif s == env.goal:
            axes[2].text(col+0.5, 3-row+0.5, '★', ha='center', va='center', fontsize=18)
        elif s in env.traps:
            axes[2].add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#e74c3c', alpha=0.2))
            axes[2].text(col+0.5, 3-row+0.5, 'T', ha='center', va='center', fontsize=14, color='red')
        elif s in env.walls:
            axes[2].add_patch(plt.Rectangle((col, 3-row), 1, 1, color='#2c3e50', alpha=0.8))
        else:
            arrow = env.action_names[np.argmax(Q_learned[s])]
            axes[2].text(col+0.5, 3-row+0.5, arrow, ha='center', va='center', fontsize=16, fontweight='bold')
axes[2].set_xlim(0, 4)
axes[2].set_ylim(0, 4)
axes[2].set_aspect('equal')
axes[2].set_title('Learned Policy')

plt.tight_layout()
plt.show()

print("\n🎯 The agent learned to navigate from S to ★ while avoiding the trap (T) and walls (■)!")

In [ ]:
# §4: Test the Q-Learning Agent

def test_agent(env, Q, n_episodes=1000):
    """Test a trained agent using greedy policy (no exploration)."""
    successes = 0
    total_steps = 0
    
    for _ in range(n_episodes):
        state = env.reset()
        done = False
        steps = 0
        
        while not done and steps < 50:
            action = np.argmax(Q[state])  # Greedy!
            state, reward, done = env.step(action)
            steps += 1
        
        if reward > 0:
            successes += 1
        total_steps += steps
    
    return successes / n_episodes, total_steps / n_episodes

success_rate, avg_steps = test_agent(env, Q_learned)
print(f"Test Results (1000 episodes, greedy policy):")
print(f"  Success rate: {success_rate:.1%}")
print(f"  Average steps to goal: {avg_steps:.1f}")

---

## §5 — SARSA (On-Policy Learning) 🟡

**SARSA** (State-Action-Reward-State-Action) is similar to Q-Learning but uses the **actual next action** instead of the **best** next action.

### Key Difference:

| Algorithm | Update Uses | Behavior |
|-----------|------------|----------|
| **Q-Learning** (off-policy) | `max Q(s',a')` — best possible | Optimistic, risk-taking |
| **SARSA** (on-policy) | `Q(s',a')` — actual next action | Conservative, risk-averse |

### When does this matter?

When there's a **cliff** (dangerous state) near the optimal path! SARSA learns a SAFER route.

In [ ]:
# §5: SARSA Implementation

def sarsa(env, n_episodes=10000, alpha=0.1, gamma=0.99,
          epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995):
    """
    SARSA — On-policy TD learning.
    Learns the value of the policy it actually follows.
    """
    Q = np.zeros((env.n_states, env.n_actions))
    epsilon = epsilon_start
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        
        # Choose first action
        if np.random.random() < epsilon:
            action = np.random.randint(env.n_actions)
        else:
            action = np.argmax(Q[state])
        
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            next_state, reward, done = env.step(action)
            
            # Choose NEXT action (actual, not best!)
            if np.random.random() < epsilon:
                next_action = np.random.randint(env.n_actions)
            else:
                next_action = np.argmax(Q[next_state])
            
            # SARSA update: uses Q(s', a') — the real next action
            td_target = reward + gamma * Q[next_state, next_action] * (1 - done)
            Q[state, action] += alpha * (td_target - Q[state, action])
            
            state = next_state
            action = next_action
            total_reward += reward
            steps += 1
        
        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, rewards_history

# Train SARSA on the same environment
env_sarsa = GridWorld()
Q_sarsa, rewards_sarsa = sarsa(env_sarsa, n_episodes=10000)

print("✅ SARSA Training Complete!")
print(f"   Final avg reward (last 100): {np.mean(rewards_sarsa[-100:]):.3f}")

In [ ]:
# §5: Compare Q-Learning vs SARSA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q-Learning vs SARSA Comparison', fontsize=16, fontweight='bold')

# Learning curves
window = 100
smooth_q = np.convolve(rewards, np.ones(window)/window, 'valid')
smooth_s = np.convolve(rewards_sarsa, np.ones(window)/window, 'valid')

axes[0].plot(smooth_q, label='Q-Learning', color='#e74c3c', linewidth=2)
axes[0].plot(smooth_s, label='SARSA', color='#2ecc71', linewidth=2)
axes[0].set_title('Learning Curves', fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward (100-ep avg)')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Success rates
sr_q, _ = test_agent(GridWorld(), Q_learned)
sr_s, _ = test_agent(GridWorld(), Q_sarsa)
axes[1].bar(['Q-Learning', 'SARSA'], [sr_q * 100, sr_s * 100],
            color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=2)
axes[1].set_ylabel('Success Rate (%)')
axes[1].set_title('Test Success Rate (1000 episodes)', fontweight='bold')
for i, v in enumerate([sr_q*100, sr_s*100]):
    axes[1].text(i, v+1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nQ-Learning success rate: {sr_q:.1%}")
print(f"SARSA success rate:      {sr_s:.1%}")
print(f"\nOn GridWorld (no dangerous cliffs), both perform similarly.")
print(f"Run 02_sarsa_comparison.py to see the Cliff Walking example")
print(f"where the difference is dramatic!")

---

## §6 — Policy Gradient Methods 🔴

Instead of learning Q-values and deriving a policy, **Policy Gradient** methods directly learn the **policy** itself.

### REINFORCE Algorithm:
1. Run a full episode
2. Calculate discounted returns for each step
3. Increase probability of actions that led to high returns
4. Decrease probability of actions that led to low returns

In [ ]:
# §6: Simplified Policy Gradient (REINFORCE)

class PolicyGradientAgent:
    """REINFORCE with softmax policy — learns policy directly."""
    
    def __init__(self, n_states, n_actions, lr=0.01, gamma=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.theta = np.zeros((n_states, n_actions))  # Policy parameters
    
    def get_action_probs(self, state):
        """Softmax policy: convert logits to probabilities."""
        logits = self.theta[state]
        exp_logits = np.exp(logits - np.max(logits))  # Numerical stability
        return exp_logits / exp_logits.sum()
    
    def choose_action(self, state):
        probs = self.get_action_probs(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def update(self, trajectory):
        """Update policy using REINFORCE (full episode required)."""
        states, actions, rewards_list = zip(*trajectory)
        
        # Calculate discounted returns (backward)
        returns = []
        G = 0
        for r in reversed(rewards_list):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = np.array(returns)
        
        # Normalize (baseline subtraction)
        if returns.std() > 0:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Update: increase prob of good actions, decrease bad
        for s, a, G_t in zip(states, actions, returns):
            probs = self.get_action_probs(s)
            for a_i in range(self.n_actions):
                if a_i == a:
                    self.theta[s, a_i] += self.lr * G_t * (1 - probs[a_i])
                else:
                    self.theta[s, a_i] -= self.lr * G_t * probs[a_i]

# Train Policy Gradient agent
pg_agent = PolicyGradientAgent(n_states=16, n_actions=4, lr=0.01, gamma=0.99)
env_pg = GridWorld()
pg_rewards = []

for episode in range(10000):
    state = env_pg.reset()
    trajectory = []
    done = False
    steps = 0
    
    while not done and steps < 100:
        action = pg_agent.choose_action(state)
        next_state, reward, done = env_pg.step(action)
        trajectory.append((state, action, reward))
        state = next_state
        steps += 1
    
    pg_agent.update(trajectory)
    pg_rewards.append(sum(r for _, _, r in trajectory))

print("✅ Policy Gradient Training Complete!")
print(f"   Final avg reward (last 100): {np.mean(pg_rewards[-100:]):.3f}")

# Compare all three methods
fig, ax = plt.subplots(figsize=(12, 5))
window = 100
for name, rews, color in [
    ('Q-Learning', rewards, '#e74c3c'),
    ('SARSA', rewards_sarsa, '#2ecc71'),
    ('Policy Gradient', pg_rewards, '#3498db'),
]:
    smoothed = np.convolve(rews, np.ones(window)/window, 'valid')
    ax.plot(smoothed, label=name, color=color, linewidth=2)

ax.set_title('Three RL Methods Compared', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## §7 — Multi-Armed Bandits & Exploration 🟢–🟡

The **Multi-Armed Bandit** is the simplest RL problem — choosing between slot machines to maximize total reward.

### The Core Dilemma: Explore vs Exploit

> **Explore**: Try new actions to discover potentially better rewards  
> **Exploit**: Use the best known action to maximize current reward  
> The challenge: **how do you balance both?**

### Three Strategies:

| Strategy | How It Works |
|----------|-------------|
| **ε-Greedy** | Random with prob ε, best known otherwise |
| **UCB** | Pick arm with highest Q + confidence bonus |
| **Thompson Sampling** | Sample from Bayesian posterior of each arm |

In [ ]:
# §7: Multi-Armed Bandit — Interactive Demo

class Bandit:
    """5-armed bandit with different reward probabilities."""
    def __init__(self):
        self.probs = [0.1, 0.3, 0.7, 0.4, 0.2]
        self.n_arms = 5
        self.best = np.argmax(self.probs)
    
    def pull(self, arm):
        return float(np.random.random() < self.probs[arm])

# ε-Greedy
class EpsilonGreedy:
    def __init__(self, n_arms, eps=0.1):
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.eps = eps
        self.name = f'ε-Greedy ({eps})'
    def select(self):
        if np.random.random() < self.eps: return np.random.randint(len(self.Q))
        return np.argmax(self.Q)
    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

# UCB
class UCB:
    def __init__(self, n_arms, c=2.0):
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.c = c
        self.t = 0
        self.name = f'UCB (c={c})'
    def select(self):
        self.t += 1
        for i in range(len(self.Q)):
            if self.N[i] == 0: return i
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb)
    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

# Thompson Sampling
class Thompson:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
        self.N = np.zeros(n_arms)
        self.name = 'Thompson'
    def select(self):
        return np.argmax(np.random.beta(self.alpha, self.beta))
    def update(self, arm, reward):
        self.N[arm] += 1
        self.alpha[arm] += reward
        self.beta[arm] += 1 - reward

# Run experiment
n_steps = 2000
n_runs = 20
strategies = [
    ('ε-Greedy (0.1)', lambda: EpsilonGreedy(5, 0.1)),
    ('UCB', lambda: UCB(5, 2.0)),
    ('Thompson', lambda: Thompson(5)),
]

results = {}
for name, factory in strategies:
    all_regrets = []
    all_optimal = []
    for _ in range(n_runs):
        bandit = Bandit()
        agent = factory()
        regrets = []
        optimals = []
        for t in range(n_steps):
            arm = agent.select()
            reward = bandit.pull(arm)
            agent.update(arm, reward)
            regrets.append(bandit.probs[bandit.best] - bandit.probs[arm])
            optimals.append(float(arm == bandit.best))
        all_regrets.append(np.cumsum(regrets))
        all_optimal.append(optimals)
    results[name] = {
        'regret': np.mean(all_regrets, axis=0),
        'optimal': np.mean(all_optimal, axis=0),
    }

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Multi-Armed Bandit: Strategy Comparison', fontsize=16, fontweight='bold')

colors = {'ε-Greedy (0.1)': '#e74c3c', 'UCB': '#3498db', 'Thompson': '#2ecc71'}
for name, res in results.items():
    ax1.plot(res['regret'], label=name, color=colors[name], linewidth=2)
ax1.set_title('Cumulative Regret (Lower = Better)')
ax1.set_xlabel('Step')
ax1.set_ylabel('Cumulative Regret')
ax1.legend()
ax1.grid(True, alpha=0.3)

for name, res in results.items():
    smoothed = np.convolve(res['optimal'], np.ones(50)/50, 'valid')
    ax2.plot(smoothed * 100, label=name, color=colors[name], linewidth=2)
ax2.set_title('% Optimal Action (Higher = Better)')
ax2.set_xlabel('Step')
ax2.set_ylabel('% Optimal')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🏆 Thompson Sampling typically wins — it balances exploration")
print("   and exploitation naturally via Bayesian uncertainty!")

---

## §8 — Real-World RL Applications 🟢

| Application | Company | Method | Impact |
|-------------|---------|--------|--------|
| Game Playing | DeepMind (AlphaGo) | MCTS + RL | Beat world champion at Go |
| Robotics | Boston Dynamics | Policy Gradient | Robot locomotion |
| Recommendations | Netflix, YouTube | Contextual Bandits | Personalized content |
| Trading | Hedge Funds | DQN | Automated strategies |
| Data Centers | Google DeepMind | DQN | 40% cooling energy reduction |
| Autonomous Driving | Waymo, Tesla | Actor-Critic | Driving decisions |
| Healthcare | IBM | Bandits | Dynamic treatment regimens |
| NLP | OpenAI (ChatGPT) | RLHF | Aligns LLMs with human preferences |

---

## 📝 Summary & Key Takeaways

### What We Covered:

1. **RL Framework**: Agent interacts with environment, receives rewards, learns policy
2. **MDP**: Mathematical formalization — (S, A, P, R, γ)
3. **Q-Learning**: Off-policy, uses `max Q(s',a')`, optimistic
4. **SARSA**: On-policy, uses actual `Q(s',a')`, conservative
5. **Policy Gradient**: Directly learns the policy, works with continuous actions
6. **Multi-Armed Bandits**: Exploration vs exploitation — ε-Greedy, UCB, Thompson

### When to Use Which:

| Scenario | Best Method |
|----------|------------|
| Simple discrete environment | Q-Learning |
| Safety-critical training | SARSA |
| Continuous action spaces | Policy Gradient |
| A/B testing / ad selection | Thompson Sampling |
| Need theoretical guarantees | UCB |

### Next Steps:
- Run the standalone scripts in `code/` for deeper analysis
- Complete the exercises in `exercises/exercises.md`
- Build the portfolio RL dashboard from `portfolio/portfolio_component.md`

---

**📚 Further Reading:**
- Sutton & Barto: *Reinforcement Learning: An Introduction* (free online)
- OpenAI Spinning Up: spinningup.openai.com
- David Silver's RL lecture series (YouTube)